# Variant 1 — No Gating (Baseline Lower Bound)
**Multi-task DistilBERT: DistilBERT → mean pool → ClassHead × 2. No target-aware gating.**

In [12]:
!pip install transformers scikit-learn pandas torch tqdm --quiet
print("Libraries installed.")

Libraries installed.


In [13]:
import os

DATA_DIR   = "/kaggle/input/datasets/ervarishitha11/datasets-inlp"
OUTPUT_DIR = "/kaggle/working"

files_needed = [
    "QAEvasion.csv",
    "raw_train_biden.csv","raw_val_biden.csv","raw_test_biden.csv",
    "raw_train_trump.csv","raw_val_trump.csv","raw_test_trump.csv",
    "raw_train_bernie.csv","raw_val_bernie.csv","raw_test_bernie.csv",
]
print("Checking files:")
all_ok = True
for f in files_needed:
    ok = os.path.exists(os.path.join(DATA_DIR, f))
    print(f"  {'OK' if ok else 'MISSING':7s}  {f}")
    if not ok: all_ok = False
print("All files found!" if all_ok else "Fix missing files.")

Checking files:
  OK       QAEvasion.csv
  OK       raw_train_biden.csv
  OK       raw_val_biden.csv
  OK       raw_test_biden.csv
  OK       raw_train_trump.csv
  OK       raw_val_trump.csv
  OK       raw_test_trump.csv
  OK       raw_train_bernie.csv
  OK       raw_val_bernie.csv
  OK       raw_test_bernie.csv
All files found!


In [14]:
import os, random, warnings, itertools
import pandas as pd, numpy as np
import torch, torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type=="cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

BERT_MODEL     = "distilbert-base-uncased"
MAX_LEN        = 256
STANCE_MAX_LEN = 128
BATCH_SIZE     = 32
EPOCHS         = 5
DROPOUT        = 0.3
WARMUP_RATIO   = 0.1
FREEZE_EPOCHS  = 2
LR_EVASION     = 1e-5
LR_STANCE      = 2e-5

VARIANT_NAME    = "Variant 1 — No Gating"
SAVE_TAG        = "v1"
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_v1_no_gating.pt")

EVASION_LABELS   = ["Non-Evasive", "Partially Evasive", "Evasive"]
EVASION_LABEL2ID = {l: i for i, l in enumerate(EVASION_LABELS)}
STANCE_LABELS    = ["FAVOR", "AGAINST"]
STANCE_LABEL2ID  = {l: i for i, l in enumerate(STANCE_LABELS)}

EVASION_CSV       = f"{DATA_DIR}/QAEvasion.csv"
STANCE_TRAIN_CSVS = [f"{DATA_DIR}/raw_train_biden.csv",f"{DATA_DIR}/raw_train_trump.csv",f"{DATA_DIR}/raw_train_bernie.csv"]
STANCE_VAL_CSVS   = [f"{DATA_DIR}/raw_val_biden.csv",  f"{DATA_DIR}/raw_val_trump.csv",  f"{DATA_DIR}/raw_val_bernie.csv"]
STANCE_TEST_CSVS  = [f"{DATA_DIR}/raw_test_biden.csv", f"{DATA_DIR}/raw_test_trump.csv", f"{DATA_DIR}/raw_test_bernie.csv"]

print(f"  Epochs:{EPOCHS}  BERT frozen first {FREEZE_EPOCHS} epochs")
print(f"  LR evasion:{LR_EVASION}  LR stance:{LR_STANCE}  Batch:{BATCH_SIZE}")
print(f"  {VARIANT_NAME}")

Device: cuda
GPU: Tesla T4
  Epochs:5  BERT frozen first 2 epochs
  LR evasion:1e-05  LR stance:2e-05  Batch:32
  Variant 1 — No Gating


In [15]:
def map_evasion_label(raw):
    raw = str(raw).strip()
    if raw.startswith("1."): return "Non-Evasive"
    elif raw == "2.3 Partial/half-answer": return "Partially Evasive"
    else: return "Evasive"

def load_evasion(filepath):
    print("Loading QA Evasion...")
    df = pd.read_csv(filepath)
    df["question"] = df["interview_question"].fillna("").str.strip()
    df["answer"]   = df["interview_answer"].fillna("").str.strip()
    df["coarse_label"] = df["label"].apply(map_evasion_label)
    df["label_id"]     = df["coarse_label"].map(EVASION_LABEL2ID).astype(int)
    df = df.dropna(subset=["question","answer","label_id"]).reset_index(drop=True)
    print(f"  Total: {len(df)}")
    for lbl in EVASION_LABELS:
        n = (df["coarse_label"]==lbl).sum()
        print(f"  {lbl:22s}: {n:4d} ({n/len(df)*100:.1f}%)")
    idx = list(range(len(df)))
    lbl = df["label_id"].tolist()
    itr,itmp,_,ytmp = train_test_split(idx,lbl,test_size=0.20,random_state=SEED,stratify=lbl)
    iva,ite,_,_     = train_test_split(itmp,ytmp,test_size=0.50,random_state=SEED,stratify=ytmp)
    q,a,l = df["question"].tolist(),df["answer"].tolist(),df["label_id"].tolist()
    def ex(ii): return [q[i] for i in ii],[a[i] for i in ii],[l[i] for i in ii]
    tr,va,te = ex(itr),ex(iva),ex(ite)
    print(f"  Train:{len(itr)}  Val:{len(iva)}  Test:{len(ite)}")
    return tr,va,te

def load_stance_split(filepaths, split_name):
    print(f"Loading Stance [{split_name}]...")
    dfs=[]
    for fp in filepaths:
        if os.path.exists(fp):
            d=pd.read_csv(fp); dfs.append(d)
            print(f"  {os.path.basename(fp):30s}  {len(d)} rows")
    df=pd.concat(dfs,ignore_index=True)
    df=df[df["Stance"].isin(STANCE_LABEL2ID)].reset_index(drop=True)
    df["label_id"]=df["Stance"].map(STANCE_LABEL2ID).astype(int)
    df["target"]=df["Target"].fillna("").str.strip()
    df["tweet"]=df["Tweet"].fillna("").str.strip()
    df=df.dropna(subset=["target","tweet","label_id"]).reset_index(drop=True)
    print(f"  Total: {len(df)}")
    return df["target"].tolist(),df["tweet"].tolist(),df["label_id"].tolist()

print("="*60)
(Qetr,Aetr,yetr),(Qeva,Aeva,yeva),(Qete,Aete,yete)=load_evasion(EVASION_CSV)
print()
Qstr,Astr,ystr=load_stance_split(STANCE_TRAIN_CSVS,"train")
print()
Qsva,Asva,ysva=load_stance_split(STANCE_VAL_CSVS,"val")
print()
Qste,Aste,yste=load_stance_split(STANCE_TEST_CSVS,"test")
print("="*60)
print("Datasets loaded.")

Loading QA Evasion...
  Total: 3448
  Non-Evasive           : 1540 (44.7%)
  Partially Evasive     :   79 (2.3%)
  Evasive               : 1829 (53.0%)
  Train:2758  Val:345  Test:345

Loading Stance [train]...
  raw_train_biden.csv             5806 rows
  raw_train_trump.csv             6362 rows
  raw_train_bernie.csv            5056 rows
  Total: 17224

Loading Stance [val]...
  raw_val_biden.csv               745 rows
  raw_val_trump.csv               814 rows
  raw_val_bernie.csv              634 rows
  Total: 2193

Loading Stance [test]...
  raw_test_biden.csv              745 rows
  raw_test_trump.csv              777 rows
  raw_test_bernie.csv             635 rows
  Total: 2157
Datasets loaded.


In [16]:
def save_split(q,a,l,lmap,fname):
    pd.DataFrame({"question":q,"answer":a,"label_id":l,
                  "label":[lmap[x] for x in l]
    }).to_csv(os.path.join(OUTPUT_DIR,fname),index=False)
    print(f"  Saved {fname}: {len(l)} rows")

save_split(Qetr,Aetr,yetr,EVASION_LABELS,"evasion_train.csv")
save_split(Qeva,Aeva,yeva,EVASION_LABELS,"evasion_val.csv")
save_split(Qete,Aete,yete,EVASION_LABELS,"evasion_test.csv")
save_split(Qstr,Astr,ystr,STANCE_LABELS,"stance_train.csv")
save_split(Qsva,Asva,ysva,STANCE_LABELS,"stance_val.csv")
save_split(Qste,Aste,yste,STANCE_LABELS,"stance_test.csv")
print("Splits saved.")

  Saved evasion_train.csv: 2758 rows
  Saved evasion_val.csv: 345 rows
  Saved evasion_test.csv: 345 rows
  Saved stance_train.csv: 17224 rows
  Saved stance_val.csv: 2193 rows
  Saved stance_test.csv: 2157 rows
Splits saved.


In [17]:
tokenizer = DistilBertTokenizer.from_pretrained(BERT_MODEL)

class TaskDataset(Dataset):
    def __init__(self, ta, tb, labels, max_len, task_id):
        self.ta,self.tb,self.labels,self.max_len,self.task_id = ta,tb,labels,max_len,task_id
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc = tokenizer(self.ta[idx],self.tb[idx],truncation=True,
                        padding="max_length",max_length=self.max_len,return_tensors="pt")
        return {"input_ids":enc["input_ids"].squeeze(0),
                "attention_mask":enc["attention_mask"].squeeze(0),
                "labels":torch.tensor(self.labels[idx],dtype=torch.long),
                "task_id":torch.tensor(self.task_id,dtype=torch.long)}

def make_loader(ta,tb,y,tid,ml,shuffle):
    return DataLoader(TaskDataset(ta,tb,y,ml,tid),batch_size=BATCH_SIZE,
                      shuffle=shuffle,num_workers=2,pin_memory=True)

loaders = {
    "evasion_train": make_loader(Qetr,Aetr,yetr,0,MAX_LEN,True),
    "evasion_val":   make_loader(Qeva,Aeva,yeva,0,MAX_LEN,False),
    "evasion_test":  make_loader(Qete,Aete,yete,0,MAX_LEN,False),
    "stance_train":  make_loader(Qstr,Astr,ystr,1,STANCE_MAX_LEN,True),
    "stance_val":    make_loader(Qsva,Asva,ysva,1,STANCE_MAX_LEN,False),
    "stance_test":   make_loader(Qste,Aste,yste,1,STANCE_MAX_LEN,False),
}
for k,v in loaders.items():
    print(f"  {k:<20} {len(v.dataset):>6,} samples  {len(v):>5,} batches")
print("DataLoaders ready.")

  evasion_train         2,758 samples     87 batches
  evasion_val             345 samples     11 batches
  evasion_test            345 samples     11 batches
  stance_train         17,224 samples    539 batches
  stance_val            2,193 samples     69 batches
  stance_test           2,157 samples     68 batches
DataLoaders ready.


In [18]:
def mean_pooling(h, mask):
    m = mask.unsqueeze(-1).expand(h.size()).float()
    return torch.sum(h*m,1) / torch.clamp(m.sum(1),min=1e-9)

class ClassHead(nn.Module):
    """Simple 2-layer MLP classifier head."""
    def __init__(self, H, n, dr):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(dr),nn.Linear(H,256),nn.ReLU(),
                                 nn.Dropout(dr),nn.Linear(256,n))
    def forward(self,x): return self.net(x)

class MultiTaskDistilBERT_NoGating(nn.Module):
    """
    Variant 1 — No Gating (baseline lower bound).
    DistilBERT shared encoder -> mean pool -> task-specific ClassHead.
    No explicit gating; question/target passed as segment A so the encoder
    sees it via self-attention only. No target-aware reweighting.
    """
    def __init__(self, ew, sw):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained(BERT_MODEL)
        H = self.encoder.config.hidden_size
        self.evasion_head    = ClassHead(H, len(EVASION_LABELS), DROPOUT)
        self.stance_head     = ClassHead(H, len(STANCE_LABELS),  DROPOUT)
        self.evasion_loss_fn = nn.CrossEntropyLoss(weight=ew)
        self.stance_loss_fn  = nn.CrossEntropyLoss(weight=sw)

    def freeze_inactive_head(self, task):
        if task==0:
            for p in self.stance_head.parameters():  p.requires_grad=False
            for p in self.evasion_head.parameters(): p.requires_grad=True
        else:
            for p in self.evasion_head.parameters(): p.requires_grad=False
            for p in self.stance_head.parameters():  p.requires_grad=True
        for p in self.encoder.parameters(): p.requires_grad=True

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad=True

    def forward(self, input_ids, attention_mask, task_id, labels=None):
        h    = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        pool = mean_pooling(h, attention_mask)   # mean over ALL tokens — no target awareness
        task = int(task_id[0].item())
        logits = self.evasion_head(pool) if task==0 else self.stance_head(pool)
        loss = None
        if labels is not None:
            fn = self.evasion_loss_fn if task==0 else self.stance_loss_fn
            loss = fn(logits, labels)
        return loss, logits

ew = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1,2]),y=yetr),dtype=torch.float).to(device)
sw = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1]),  y=ystr),dtype=torch.float).to(device)
print(f"Evasion weights: {ew.tolist()}")
print(f"Stance  weights: {sw.tolist()}")
model = MultiTaskDistilBERT_NoGating(ew, sw).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print("Arch: DistilBERT -> mean pool (all tokens) -> ClassHead x2  [NO target-aware gating]")

Evasion weights: [0.7462121248245239, 14.592592239379883, 0.6283891797065735]
Stance  weights: [1.0317479372024536, 0.9701475501060486]


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parameters: 66,757,893
Arch: DistilBERT -> mean pool (all tokens) -> ClassHead x2  [NO target-aware gating]


In [19]:
@torch.no_grad()
def evaluate(loader, label_names):
    model.eval(); model.unfreeze_all()
    all_preds, all_labels, total_loss = [], [], 0.0
    for batch in loader:
        loss, logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["task_id"].to(device),
            batch["labels"].to(device))
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits,1).cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
    return {
        "loss":   total_loss/len(loader),
        "acc":    accuracy_score(all_labels,all_preds),
        "f1":     f1_score(all_labels,all_preds,average="macro",zero_division=0),
        "prec":   precision_score(all_labels,all_preds,average="macro",zero_division=0),
        "rec":    recall_score(all_labels,all_preds,average="macro",zero_division=0),
        "report": classification_report(all_labels,all_preds,target_names=label_names,zero_division=0),
        "cm":     pd.DataFrame(confusion_matrix(all_labels,all_preds,labels=list(range(len(label_names)))),
                    index=[f"True:{l}" for l in label_names],
                    columns=[f"Pred:{l}" for l in label_names]),
        "preds": all_preds, "labels": all_labels,
    }

def print_eval(m, title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(f"  Loss:{m['loss']:.4f}  Acc:{m['acc']*100:.2f}%  MacroF1:{m['f1']:.4f}  Prec:{m['prec']:.4f}  Rec:{m['rec']:.4f}")
    print("\n  Per-class Report:")
    for line in m["report"].strip().split("\n"): print(f"  {line}")
    print("\n  Confusion Matrix:")
    print(m["cm"].to_string())
    print("="*60)

print("Evaluation functions ready.")

Evaluation functions ready.


In [20]:
optimizer = AdamW([
    {"params": list(model.evasion_head.parameters()), "lr": LR_EVASION},
    {"params": list(model.stance_head.parameters()),  "lr": LR_STANCE},
    {"params": list(model.encoder.parameters()),      "lr": LR_EVASION},
], weight_decay=0.01)

n_steps  = len(loaders["stance_train"]) * 3 * EPOCHS
n_warmup = int(WARMUP_RATIO * n_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, n_warmup, n_steps)
print(f"Optimizer ready. Steps: {n_steps}  Warmup: {n_warmup}")

Optimizer ready. Steps: 8085  Warmup: 808


In [21]:
import itertools

best_avg_f1 = 0.0
history     = []

print("="*60)
print(f"  TRAINING — {VARIANT_NAME}")
print(f"  Epochs: {EPOCHS}  BERT frozen first {FREEZE_EPOCHS} epochs")
print(f"  LR evasion: {LR_EVASION}  LR stance: {LR_STANCE}  Batch: {BATCH_SIZE}")
print("="*60)

for p in model.encoder.parameters():
    p.requires_grad = False
print(f"BERT encoder FROZEN for epochs 1-{FREEZE_EPOCHS}.")

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_EPOCHS + 1:
        for p in model.encoder.parameters():
            p.requires_grad = True
        print(f"\n  Epoch {epoch}: BERT UNFROZEN — full fine-tuning begins.")

    model.train()
    run_e_loss = run_s_loss = 0.0
    n_e = n_s = 0

    e_cycle  = itertools.cycle(list(loaders["evasion_train"]))
    combined = []
    for s_batch in loaders["stance_train"]:
        combined.append(next(e_cycle))
        combined.append(next(e_cycle))
        combined.append(s_batch)

    total = len(combined)
    print(f"\nEpoch {epoch}/{EPOCHS} — {total} steps")

    for step, batch in enumerate(combined, 1):
        task = int(batch["task_id"][0].item())
        model.freeze_inactive_head(task)
        optimizer.zero_grad()
        loss, logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["task_id"].to(device),
            batch["labels"].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        if task == 0: run_e_loss += loss.item(); n_e += 1
        else:         run_s_loss += loss.item(); n_s += 1
        done = int(30 * step / total)
        print(f"\r  [{'='*done}{'-'*(30-done)}] {step}/{total}", end="", flush=True)

    avg_e = run_e_loss / n_e if n_e else 0
    avg_s = run_s_loss / n_s if n_s else 0
    print()

    print("  Validating...")
    e_val  = evaluate(loaders["evasion_val"], EVASION_LABELS)
    s_val  = evaluate(loaders["stance_val"],  STANCE_LABELS)
    avg_f1 = (e_val["f1"] + s_val["f1"]) / 2
    is_best = avg_f1 > best_avg_f1

    print(f"\n  Epoch {epoch}/{EPOCHS} Summary")
    print(f"  {'-'*52}")
    print(f"  {'Metric':<24} {'Evasion':>10} {'Stance':>10}")
    print(f"  {'-'*52}")
    print(f"  {'Train Loss':<24} {avg_e:>10.4f} {avg_s:>10.4f}")
    print(f"  {'Val Loss':<24} {e_val['loss']:>10.4f} {s_val['loss']:>10.4f}")
    print(f"  {'Accuracy':<24} {e_val['acc']*100:>9.2f}% {s_val['acc']*100:>9.2f}%")
    print(f"  {'Macro F1':<24} {e_val['f1']:>10.4f} {s_val['f1']:>10.4f}")
    print(f"  {'Precision':<24} {e_val['prec']:>10.4f} {s_val['prec']:>10.4f}")
    print(f"  {'Recall':<24} {e_val['rec']:>10.4f} {s_val['rec']:>10.4f}")
    print(f"  {'-'*52}")
    print(f"  Avg Macro F1: {avg_f1:.4f}  {'<- BEST' if is_best else ''}")

    if is_best:
        best_avg_f1 = avg_f1
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  Saved best model -> {BEST_MODEL_PATH}")

    history.append({"epoch":epoch,"e_f1":e_val["f1"],"s_f1":s_val["f1"],"avg_f1":avg_f1})

print("\n" + "="*60)
print(f"  Training complete.  Best Avg Macro F1: {best_avg_f1:.4f}")
print("="*60)

  TRAINING — Variant 1 — No Gating
  Epochs: 5  BERT frozen first 2 epochs
  LR evasion: 1e-05  LR stance: 2e-05  Batch: 32
BERT encoder FROZEN for epochs 1-2.

Epoch 1/5 — 1617 steps
  [==============================] 1617/1617
  Validating...

  Epoch 1/5 Summary
  ----------------------------------------------------
  Metric                      Evasion     Stance
  ----------------------------------------------------
  Train Loss                   0.9640     0.6102
  Val Loss                     1.5308     0.5470
  Accuracy                     58.55%     72.05%
  Macro F1                     0.4316     0.7170
  Precision                    0.4377     0.7242
  Recall                       0.4325     0.7171
  ----------------------------------------------------
  Avg Macro F1: 0.5743  <- BEST
  Saved best model -> /kaggle/working/best_v1_no_gating.pt

Epoch 2/5 — 1617 steps
  [==============================] 1617/1617
  Validating...

  Epoch 2/5 Summary
  ---------------------------

In [22]:
print(f"Loading best model: {BEST_MODEL_PATH}")
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.unfreeze_all()
print("Model loaded.\n")

e_test = evaluate(loaders["evasion_test"], EVASION_LABELS)
s_test = evaluate(loaders["stance_test"],  STANCE_LABELS)

print_eval(e_test, f"QA EVASION — Test  [{VARIANT_NAME}]")
print_eval(s_test, f"STANCE    — Test  [{VARIANT_NAME}]")

pd.DataFrame({"question":Qete,"answer":Aete,
    "true":[EVASION_LABELS[l] for l in e_test["labels"]],
    "pred":[EVASION_LABELS[p] for p in e_test["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,f"results_{SAVE_TAG}_evasion.csv"),index=False)

pd.DataFrame({"target":Qste,"tweet":Aste,
    "true":[STANCE_LABELS[l] for l in s_test["labels"]],
    "pred":[STANCE_LABELS[p] for p in s_test["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,f"results_{SAVE_TAG}_stance.csv"),index=False)

print(f"\n{'='*60}")
print(f"  {VARIANT_NAME} — Final Results")
print(f"  Evasion Macro F1 : {e_test['f1']:.4f}")
print(f"  Stance  Macro F1 : {s_test['f1']:.4f}")
print(f"  Avg Macro F1     : {(e_test['f1']+s_test['f1'])/2:.4f}")
print(f"{'='*60}")
print("Files saved to OUTPUT_DIR.")

Loading best model: /kaggle/working/best_v1_no_gating.pt
Model loaded.


  QA EVASION — Test  [Variant 1 — No Gating]
  Loss:2.2594  Acc:60.87%  MacroF1:0.4592  Prec:0.4661  Rec:0.4553

  Per-class Report:
  precision    recall  f1-score   support
  
        Non-Evasive       0.58      0.62      0.60       154
  Partially Evasive       0.17      0.12      0.14         8
            Evasive       0.65      0.62      0.63       183
  
           accuracy                           0.61       345
          macro avg       0.47      0.46      0.46       345
       weighted avg       0.61      0.61      0.61       345

  Confusion Matrix:
                        Pred:Non-Evasive  Pred:Partially Evasive  Pred:Evasive
True:Non-Evasive                      96                       3            55
True:Partially Evasive                 2                       1             5
True:Evasive                          68                       2           113

  STANCE    — Test  [Variant 1 — No Gating

In [23]:
print("All done.")

All done.
